In [ ]:
# R2_CAR171_Canopus.ipynb
# Estimating the stray-light for the Commissioning Activity Request (CAR) 171 observations. 
# This notebook is based on the R1_Rosalia_stray_example.ipynb notebook, but it is adapted to the specific case of Canopus.
# - Alejandro S. Borlaff / NASA Ames / a.s.borlaff@nasa.gov. March 11, 2026.
import os
import rosalia as rs
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time
import astropy.units as u
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad
print("ROSALIA version:", rs.__version__)
plt.style.use(os.path.dirname(rs.__file__) + "/style/nature_style.mplstyle")

In [ ]:
# Let's find Canopus (alpha Carinae), Eta Uma, and HD128998, the stars selected for CAR171.
# These stars are in the continuous viewing zone of Roman. Canopus is the second brightest star in the sky.

# Find the Calibration Stars. 
alfCar = Simbad.query_object('Canopus')
ra_alfCar = alfCar["ra"][0]
dec_alfCar = alfCar["dec"][0]

etaUma = Simbad.query_object('Eta Uma')
ra_etaUma = etaUma["ra"][0]
dec_etaUma = etaUma["dec"][0]

HD = Simbad.query_object('HD128998')
ra_HD = HD["ra"][0]
dec_HD = HD["dec"][0]



# Test
target = SkyCoord(ra_alfCar*u.deg, dec_alfCar*u.deg, frame="icrs")

date = Time('2026-10-25T00:00:00.0', format='isot', scale='utc')

offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=ra_alfCar,
                                                      dec_target=dec_alfCar,
                                                      mjd=date.mjd,
                                                      dX=1, dY=1)

In [ ]:
# Equatorial pole Test
from astropy.coordinates import get_body
# date = Time('2026-10-25T00:00:00.0', format='isot', scale='utc')

sun_coord = get_body('Sun', date)   # get coordinate object for the Sun for each day of the year

ecliptic_pole = SkyCoord(0*u.deg, 89*u.deg, frame="barycentricmeanecliptic")
ecliptic_pole = ecliptic_pole.transform_to("icrs")
bestPA_for_canopus_Dec26 = rs.telescopes.Roman.get_bestPA(ra=ra_alfCar, dec=dec_alfCar, mjd=date.mjd)
print(bestPA_for_canopus_Dec26)
offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=ecliptic_pole.ra.degree,
                                                      dec_target=ecliptic_pole.dec.degree,
                                                      mjd=date.mjd,
                                                      dX=0, dY=0)
print(offset_pointing)
print(sun_coord.barycentricmeanecliptic)

In [ ]:
sun_ra = sun_coord.ra.degree
sun_dec = sun_coord.dec.degree
plt.scatter(sun_ra, sun_dec, marker="s", s=100, label="Sun")
plt.scatter(ra_alfCar, dec_alfCar, marker="o", label = "Canopus")
plt.scatter(ra_etaUma, dec_etaUma, marker="o", label = "Eta Uma")
plt.scatter(ra_HD, dec_HD, marker="o", label = "HD128998")
plt.xlim((0,360))
plt.ylim((-90,90))
plt.legend()

In [ ]:
CAR171 = pd.read_csv("CAR171_v2_coordinates.csv")
# These are the coordinates of specially sensitive locations 
# for stray-light, measured in offset degrees from the center of WFI focal plane
dX = CAR171["MPA_ThetaX"]
dY = CAR171["MPA_ThetaY"]
CAR171["RA_Source"] = 9999.
CAR171["Dec_Source"] = 9999.
CAR171.at[12:,"RA_Source"] = ra_alfCar
CAR171.at[12:,"Dec_Source"] = dec_alfCar
CAR171.at[12:,"Target source"] = "Canopus"
CAR171.at[0:4, "RA_Source"] = ra_etaUma
CAR171.at[0:4,"Dec_Source"] = dec_etaUma
CAR171.at[0:4,"Target source"] = "EtaUma"
CAR171.at[4:6,"RA_Source"] = ra_HD
CAR171.at[4:6,"Dec_Source"] = dec_HD
CAR171.at[4:6,"Target source"] = "HD128998"
CAR171.at[6:10,"RA_Source"] = ra_etaUma
CAR171.at[6:10,"Dec_Source"] = dec_etaUma
CAR171.at[6:10,"Target source"] = "EtaUma"
CAR171.at[10:11,"RA_Source"] = ra_HD
CAR171.at[10:11,"Dec_Source"] = dec_HD
CAR171.at[10:11,"Target source"] = "HD128998"

CAR171["MA Table"] = "IM_XXX_XX"
CAR171.at[0:11, "MA Table"] = "IM_171_10"
CAR171.at[12:,  "MA Table"] = "IM_193_11"

CAR171["Resultant"] = 9999
CAR171.at[0:11, "Resultant"] = 10
CAR171.at[12:,  "Resultant"] = 11

CAR171[0:16]

In [ ]:
# ra_source = CAR171["RA_Source"]
# dec_source = CAR171["Dec_Source"]
number_of_points_of_interest = len(dX)

# To measure the stray-light from those sources, we need to place the center 
# of Roman / WFI at a certain distance and position angle from the source 
# that generates the stray-light. We will name that source the "offending" source. 

# ROSALIA has a specific tool to compute those locations. 

# The optimal position angle of Roman depends with time. 
# Let's set an approximate time for the Commissioning Activities
# date = Time('2026-10-25T00:00:00.0', format='isot', scale='utc')

ra_wfi = np.zeros(number_of_points_of_interest)
dec_wfi = np.zeros(number_of_points_of_interest)
V3PA_ori = np.zeros(number_of_points_of_interest)
V3PA_off = np.zeros(number_of_points_of_interest)
WFIPA_off = np.zeros(number_of_points_of_interest)

activity_name_apt = []
category = []
description = []
for i in range(number_of_points_of_interest):
    offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=CAR171["RA_Source"].iloc[i],
                                                      dec_target=CAR171["Dec_Source"].iloc[i],
                                                      mjd=date.mjd,
                                                      dX=dX[i], dY=dY[i])
    ra_wfi[i] = offset_pointing["ra_wficen"]
    dec_wfi[i] = offset_pointing["dec_wficen"]
    WFIPA_off[i] = offset_pointing["PA_WFI_offset"]%360
    V3PA_off[i] = offset_pointing["V3PA_offset"]%360
    V3PA_ori[i] = offset_pointing["V3PA_origin"]%360
    # v3pa[i] = offset_pointing["PA_v3"] % 360
    activity_name_apt.append(CAR171["Activity name"].iloc[i]+"_"+str(i+1).zfill(3))
    category.append("Calibration")
    description.append("Stray light test")

car171_APT = pd.DataFrame({"Target": activity_name_apt, "ArchiveTarget": activity_name_apt,
                            "description":description, "category": category, "RA": ra_wfi, "DEC": dec_wfi, 
                           "WFIPA_off": WFIPA_off, "V3PA_off": V3PA_off, "V3PA_ori": V3PA_ori,
                           "CAR": CAR171["CAR"], "Region": CAR171["Region"], 
                           "Target_Source": CAR171["Target source"],
                           "RA_Source": CAR171["RA_Source"],"Dec_Source": CAR171["Dec_Source"],
                           "MPA_ThetaX": CAR171["MPA_ThetaX"], "MPA_ThetaY": CAR171["MPA_ThetaY"], 
                           "N_FRAME": CAR171["N_FRAME"], "N_EXP": CAR171["N_EXP"], 
                           "N_FILT": CAR171["N_FILT"], "Filter": CAR171["Filter(s)"],
                           "MA Table": CAR171["MA Table"], "Resultant": CAR171["Resultant"]})
car171_APT.to_csv("CAR171_apt_targets.csv")


In [5]:
import pandas as pd
star_catalog = pd.read_csv("/Users/aborlaff/NASA/ROSALIA/notebooks/COM/r0104301001001006001_0001_wfi11_f146_cal_source_catalog.csv")
bright_star_catalog = star_catalog[star_catalog["source_id"] == "Hipparcos bright star"]


/var/folders/s4/f7nrp0c95f5ccmwc7csdxt5r0000gq/T/ipykernel_58177/510933767.py:2: DtypeWarning: Columns (0: source_id) have mixed types. Specify dtype option on import or set low_memory=False.
  star_catalog = pd.read_csv("/Users/aborlaff/NASA/ROSALIA/notebooks/COM/r0104301001001006001_0001_wfi11_f146_cal_source_catalog.csv")


In [6]:
bright_star_catalog

,Unnamed: 0.2,healpix_lvl,source_id,ra,dec,phot_g_mean_mag_AB,phot_bp_mean_mag_AB,phot_rp_mean_mag_AB,phot_j_mean_mag_AB,phot_h_mean_mag_AB,...,j_error,h,h_error,ks,ks_error,g,bp,rp,mag_lambda,cat_id
4,53,NaN,Hipparcos bright star,88.792870,7.407036,0.605,0.807572,-0.094732,-2.099,-2.647,...,NaN,41571.912068,NaN,37948.967126,NaN,2079.696687,1725.723527,3961.803579,-2.188789,5
6,161,NaN,Hipparcos bright star,247.351940,-26.431946,1.205,1.526481,0.385972,-1.960,-2.365,...,NaN,32062.693245,NaN,29376.496520,NaN,1196.740531,890.036055,2544.552173,-2.026358,7
7,34,NaN,Hipparcos bright star,69.190320,-62.076980,5.705,5.947572,3.772714,-1.762,-2.372,...,NaN,32270.082820,NaN,33021.752651,NaN,18.967059,15.169498,112.438323,-1.861947,8
8,172,NaN,Hipparcos bright star,258.661930,14.390253,2.905,3.037316,1.787861,-1.412,-1.864,...,NaN,20211.567670,NaN,17076.543155,NaN,250.034536,221.347020,699.609471,-1.486059,9
9,134,NaN,Hipparcos bright star,213.918120,19.187270,0.005,0.229694,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,3614.098626,2938.476596,NaN,-1.350009,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10861,168,NaN,Hipparcos bright star,252.967670,-38.047330,3.105,2.850324,3.426660,4.348,4.842,...,NaN,41.995233,NaN,22.366601,NaN,207.969669,262.948392,154.645099,4.428941,10862
11002,153,NaN,Hipparcos bright star,239.713000,-26.114042,3.005,2.746291,3.354349,4.355,4.860,...,NaN,41.304750,NaN,22.573551,NaN,228.034207,289.389947,165.295306,4.437743,11003
11058,31,NaN,Hipparcos bright star,59.463420,40.010273,3.005,2.750622,3.350726,4.341,4.955,...,NaN,37.844258,NaN,22.019122,NaN,228.034207,288.237897,165.847802,4.441603,11059
11714,46,NaN,Hipparcos bright star,83.858250,-5.909900,2.905,2.597340,3.286696,4.380,5.008,...,NaN,36.041263,NaN,21.242219,NaN,250.034536,331.943402,175.922634,4.482896,11715


In [7]:

# if True:
for i in range(len(ra_wfi)):
    prefix = "CAR171_" + car171_APT["Target"].iloc[i]

    if os.path.exists(prefix + "/"):
        print("Directory " + prefix + " already exists. Skipping this target.")
        continue

    #i = 0
    ra = car171_APT["RA"].iloc[i] # ra_wfi[i]
    dec = car171_APT["DEC"].iloc[i] # dec_wfi[i]
    pa = car171_APT["WFIPA_off"].iloc[i] # WFIPA_off[i]
    mjd = date.mjd
    bandpass="F158"
    exptime=192

    if car171_APT["Target_Source"].iloc[i] == "Canopus":
        mag_star = -0.74
    if car171_APT["Target_Source"].iloc[i] == "EtaUma":
        mag_star = 3
    if car171_APT["Target_Source"].iloc[i] == "HD128998":
        mag_star = 6.75

    import pandas as pd
    star_catalog = pd.DataFrame({"ra": [car171_APT["RA_Source"].iloc[i]]*2, 
                                "dec": [car171_APT["Dec_Source"].iloc[i]]*2 , 
                                "source_id": 2*[car171_APT["Target_Source"].iloc[i]], 
                                "cat_id": [1,2],
                                "mag_lambda": [mag_star, 999]}, index=[0,1])

    #rosalia_stray = rs.correct.rosalia_stray(ra=ra, dec=dec, prefix="CAR171_" + CAR171["Activity name"].iloc[i] + "_",
    #                                         PA=pa, date=date, bandpass=bandpass, 
    #                                         exptime=exptime, radius=1, catalog=star_catalog,
    #                                         g_mag_max=17, verbose=3)
    observer={"TELESCOP": "Roman/WFI", "pointing": [ra, dec], 
              "FILTER":bandpass, "PA_Y": pa, "EXPSTART": mjd, "EXPTIME": exptime}
    custom_roman_exposure = rs.core.exposure(observer=observer, prefix=prefix) 
    # straylight_out = custom_roman_exposure.straylight() # catalog=star_catalog)

    
    straylight_out = custom_roman_exposure.straylight(catalog=bright_star_catalog)    
    os.system("mkdir " + prefix)
    os.system("mv " + prefix + "*.* " + prefix + "/")


NameError: name 'ra_wfi' is not defined

In [ ]:
star_catalog 

In [ ]:
car171_APT

In [ ]:
# For the Dragon's Breath example, let's change the target to a dimmer star. 
# Magnitude 2 star. 

# Let's find Canopus (alpha Carinae), the star selected for CAR171. 
# This star is in the continuous viewing zone of Roman, and it is the second brightest star in the sky.
import astropy.units as u
from astroquery.simbad import Simbad
from astropy.coordinates import SkyCoord

alfUrsae = Simbad.query_object('Mizar')
ra_star = alfUrsae["ra"][0]
dec_star = alfUrsae["dec"][0]

target = SkyCoord(ra_star*u.deg, dec_star*u.deg, frame="icrs")
print(target)

# These are the coordinates of specially sensitive locations 
# for stray-light, measured in offset degrees from the center of WFI focal plane
dX = np.array([-0.0688, 0.0688, -0.138, 0.138])
dY = np.array([-0.0312-20/60/60, -0.0312, -0.0035, -0.0048])
number_of_points_of_interest = len(dX)


# To measure the stray-light from those sources, we need to place the center 
# of Roman / WFI at a certain distance and position angle from the source 
# that generates the stray-light. We will name that source the "offending" source. 

# ROSALIA has a specific tool to compute those locations. 

# The optimal position angle of Roman depends with time. 
# Let's set an approximate time for the Commissioning Activities
from astropy.time import Time
# date = Time('2026-11-21T00:00:00.0', format='isot', scale='utc')

ra_wfi = np.zeros(number_of_points_of_interest)
dec_wfi = np.zeros(number_of_points_of_interest)
pa_wfi = np.zeros(number_of_points_of_interest)

for i in range(number_of_points_of_interest):
    offset_pointing = rs.telescopes.Roman.find_wfi_center_for_offset_target(ra_target=target.ra.degree,
                                                      dec_target=target.dec.degree,
                                                      mjd=date.mjd,
                                                      dX=dX[i], dY=dY[i])
    ra_wfi[i] = offset_pointing["ra_wficen"]
    dec_wfi[i] = offset_pointing["dec_wficen"]
    pa_wfi[i] = offset_pointing["PA_WFI_offset"]


In [ ]:
offset_pointing